# Task 1: Build a Complete ETL Pipeline
Build an ETL pipeline that processes raw e-commerce order data into a clean analytical dataset.

Step 1 — Generate raw data: Create a list of 200 raw order records as dictionaries with these fields: order_id, customer_id, product_name, quantity, unit_price, order_date, shipping_country. Intentionally include at least 15 problematic records across these categories:

3 records with missing product_name (None or empty string)
3 records with negative quantity or unit_price
3 records with malformed order_date (e.g., "not-a-date", "2025-13-45")
3 records with duplicate order_id values
3 records where quantity or unit_price is a string instead of a number

In [92]:
import random
import pandas as pd
import numpy as np
import pyarrow
from datetime import datetime, timedelta

In [94]:
random.seed(42)
products = ["Blender", "Textbooks", "Tablet", "Headphones", "Refrigerator", "Sneakers"]
countries = ["USA", "China", "Germany", "France", "Canada", "Japan"]

raw_orders = []

for i in range(1, 186):
    record = {
        "order_id": i,
        "customer_id": random.randint(1000, 1100),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(20, 500), 2),
        "order_date": (datetime(2025, 1, 1) + timedelta(days=random.randint(0, 60))).strftime("%Y-%m-%d"),
        "shipping_country": random.choice(countries)
    }
    raw_orders.append(record)
    
# 3 records with missing product_name
for i in range(186, 189):
    raw_orders.append({
        "order_id": i,
        "customer_id": random.randint(1000,1100),
        "product_name": None,
        "quantity": 2,
        "unit_price": 100,
        "order_date": "2025-02-10",
        "shipping_country": random.choice(countries)
    })

def add_orders(records):
    for r in records:
        raw_orders.append({
            "order_id": r[0],
            "customer_id": r[1],
            "product_name": random.choice(products),
            "quantity": r[2],
            "unit_price": r[3],
            "order_date": r[4],
            "shipping_country": random.choice(countries)
        })

# negative quantity or price
add_orders([
(189,1010,-1,900,"2025-02-05"),
(190,1011,2,-500,"2025-02-06"),
(191,1012,-3,300,"2025-02-07")
])

# malformed dates
add_orders([
(192,1013,1,25,"not-a-date"),
(193,1014,2,70,"2025-13-45"),
(194,1015,1,1000,"2025/99/99")
])

# duplicate order_id
add_orders([
(10,1016,1,500,"2025-02-11"),
(20,1017,2,300,"2025-02-12"),
(30,1018,3,25,"2025-02-13")
])

# quantity or price as string
add_orders([
(195,1019,"3",900,"2025-02-14"),
(196,1020,2,"700","2025-02-15"),
(197,1021,"five","300","2025-02-16")
])


# Ensure total is 200
while len(raw_orders) < 200:
    i = len(raw_orders) + 1
    raw_orders.append({
        "order_id": i,
        "customer_id": random.randint(1000,1100),
        "product_name": random.choice(products),
        "quantity": random.randint(1,5),
        "unit_price": round(random.uniform(20,500),2),
        "order_date": "2025-02-20",
        "shipping_country": random.choice(countries)
    })

print(len(raw_orders))
print(raw_orders[:5])

200
[{'order_id': 1, 'customer_id': 1081, 'product_name': 'Blender', 'quantity': 1, 'unit_price': 375.94, 'order_date': '2025-01-16', 'shipping_country': 'China'}, {'order_id': 2, 'customer_id': 1017, 'product_name': 'Sneakers', 'quantity': 1, 'unit_price': 344.82, 'order_date': '2025-02-27', 'shipping_country': 'Canada'}, {'order_id': 3, 'customer_id': 1011, 'product_name': 'Refrigerator', 'quantity': 4, 'unit_price': 35.26, 'order_date': '2025-01-06', 'shipping_country': 'China'}, {'order_id': 4, 'customer_id': 1029, 'product_name': 'Refrigerator', 'quantity': 5, 'unit_price': 32.74, 'order_date': '2025-01-13', 'shipping_country': 'Japan'}, {'order_id': 5, 'customer_id': 1083, 'product_name': 'Sneakers', 'quantity': 5, 'unit_price': 221.37, 'order_date': '2025-01-29', 'shipping_country': 'Canada'}]


### Step 2 — Extract with validation: Write an extract(raw_records) function that:

Parses each record and validates required fields exist
Validates that quantity and unit_price are positive numbers
Validates that order_date is a parseable date
Returns two lists: valid_records and rejected_records
Each rejected record should include the reason for rejection
Print a summary: count of valid, count of rejected by reason

In [97]:
def extract(raw_records):
    valid_records = []
    rejected_records = []
    reason_count = {}

    for record in raw_records:
        reason = None

        # Check required fields
        required = ["order_id","customer_id","product_name","quantity","unit_price","order_date","shipping_country"]
        for field in required:
            if field not in record or record[field] in [None, ""]:
                reason = f"missing_{field}"
                break

        # Validate quantity and unit_price
        if not reason:
            q = record["quantity"]
            p = record["unit_price"]
        
            if not isinstance(q, (int, float)) or not isinstance(p, (int, float)):
                reason = "non_numeric_quantity_or_price"
            elif q <= 0 or p <= 0:
                reason = "negative_or_zero_value"

        # Validate order_date
        if not reason:
            try:
                datetime.strptime(record["order_date"], "%Y-%m-%d")
            except:
                reason = "invalid_date"

        # Save results
        if reason:
            rejected_records.append({**record, "reason": reason})
            reason_count[reason] = reason_count.get(reason, 0) + 1
        else:
            valid_records.append(record)

    # Print summary
    print("Valid records:", len(valid_records))
    print("Rejected records:", len(rejected_records))
    print("Rejected by reason:")
    for r, c in reason_count.items():
        print(r, ":", c)

    return valid_records, rejected_records

In [99]:
valid_records, rejected_records = extract(raw_orders)

Valid records: 188
Rejected records: 12
Rejected by reason:
missing_product_name : 3
negative_or_zero_value : 3
invalid_date : 3
non_numeric_quantity_or_price : 3


### Step 3 — Transform: Write a transform(valid_records) function that:

Converts to a pandas DataFrame
Computes total_amount = quantity * unit_price
Extracts order_month and order_day_of_week from the date
Standardizes shipping_country to title case
Removes duplicate order_id values (keep the first)
Adds an amount_category column: "small" (<$25), "medium" ($25-$100), "large" (>$100)
Returns the cleaned DataFrame

In [102]:
def transform(valid_records):

    # Convert to DataFrame
    df = pd.DataFrame(valid_records)

     # Compute total_amount
    df["total_amount"] = df["quantity"] * df["unit_price"]

    # Convert order_date to datetime
    df["order_date"] = pd.to_datetime(df["order_date"])

    # Extract date features
    df["order_month"] = df["order_date"].dt.month_name()
    df["order_day_of_week"] = df["order_date"].dt.day_name()

    # Standardize shipping_country
    df["shipping_country"] = df["shipping_country"].str.title()

    # Remove duplicate order_id (keep first)
    df = df.drop_duplicates(subset="order_id", keep="first")

    # Amount category
    def categorize(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(categorize)

    return df

In [104]:
clean_df = transform(valid_records)
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 185 entries, 0 to 184
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           185 non-null    int64         
 1   customer_id        185 non-null    int64         
 2   product_name       185 non-null    object        
 3   quantity           185 non-null    int64         
 4   unit_price         185 non-null    float64       
 5   order_date         185 non-null    datetime64[ns]
 6   shipping_country   185 non-null    object        
 7   total_amount       185 non-null    float64       
 8   order_month        185 non-null    object        
 9   order_day_of_week  185 non-null    object        
 10  amount_category    185 non-null    object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 17.3+ KB


In [106]:
clean_df.head()

,order_id,customer_id,product_name,quantity,unit_price,order_date,shipping_country,total_amount,order_month,order_day_of_week,amount_category
0,1,1081,Blender,1,375.94,2025-01-16,China,375.94,January,Thursday,large
1,2,1017,Sneakers,1,344.82,2025-02-27,Canada,344.82,February,Thursday,large
2,3,1011,Refrigerator,4,35.26,2025-01-06,China,141.04,January,Monday,large
3,4,1029,Refrigerator,5,32.74,2025-01-13,Japan,163.70,January,Monday,large
4,5,1083,Sneakers,5,221.37,2025-01-29,Canada,1106.85,January,Wednesday,large


### Step 4 — Load: Write a load(df, path) function that saves the DataFrame to a Parquet file. Then read it back and verify the row count and dtypes match.

In [109]:
def load(df, path):
    
    # Save to Parquet
    df.to_parquet(path, index=False)
    
    # Read it back
    loaded_df = pd.read_parquet(path)
    
    # Verify row count
    print("Original rows:", len(df))
    print("Loaded rows:", len(loaded_df))
    
    # Verify dtypes
    print("\nOriginal dtypes:")
    print(df.dtypes)
    
    print("\nLoaded dtypes:")
    print(loaded_df.dtypes)
    return loaded_df

In [111]:
loaded_df = load(clean_df, "e_commerce.parquet")

Original rows: 185
Loaded rows: 185

Original dtypes:
order_id                      int64
customer_id                   int64
product_name                 object
quantity                      int64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                  object
order_day_of_week            object
amount_category              object
dtype: object

Loaded dtypes:
order_id                      int64
customer_id                   int64
product_name                 object
quantity                      int64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                  object
order_day_of_week            object
amount_category              object
dtype: object


### Step 5: Run the full pipeline: extract → transform → load. Print a summary showing records at each stage (raw → valid → transformed → loaded).

In [114]:
# Pipeline summary
print("\nPipeline Summary")
print("Raw records:", len(raw_orders))
print("Valid records:", len(valid_records))
print("Transformed rows:", len(clean_df))
print("Loaded rows:", len(loaded_df))


Pipeline Summary
Raw records: 200
Valid records: 188
Transformed rows: 185
Loaded rows: 185


# Task 2: ETL vs ELT Comparison
### Step 1: Implement an ELT variant of the pipeline from Task 1:

Extract: do minimal validation (only reject unparseable JSON/records)
Load: save the raw (mostly unvalidated) data to a Parquet "data lake" file
Transform: read from the data lake file, apply the same validation and transformation rules as Task 1

In [117]:
def extract_elt(raw_records):
    valid_raw = []
    rejected = []

    for r in raw_records:
        if isinstance(r, dict):
            valid_raw.append(r)
        else:
            rejected.append({"record": r, "reason": "invalid_record_format"})

    print("Extracted records:", len(valid_raw))
    print("Rejected records:", len(rejected))

    return valid_raw, rejected

In [119]:
def load_raw_to_lake(records, path):
    
    df = pd.DataFrame(records)

    # Convert all columns to string so Parquet accepts mixed raw data
    df = df.astype(str)

    df.to_parquet(path, index=False)

    print("Raw data saved to:", path)
    print("Rows saved:", len(df))

In [121]:
def transform_from_lake(path):

    df = pd.read_parquet(path)

    # Convert numeric fields
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    # Parse date
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    # Remove invalid rows
    df = df.dropna(subset=["product_name", "quantity", "unit_price", "order_date"])
    df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]

    # Transformations
    df["total_amount"] = df["quantity"] * df["unit_price"]
    df["order_month"] = df["order_date"].dt.month_name()
    df["order_day_of_week"] = df["order_date"].dt.day_name()

    df["shipping_country"] = df["shipping_country"].str.title()

    # Remove duplicates
    df = df.drop_duplicates(subset="order_id", keep="first")

    # Amount category
    def category(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(category)

    return df

### Step 2: Compare ETL and ELT approaches. Write a markdown cell addressing:

How many records made it to the destination in each approach?
At what stage were problems caught in each?
What are the advantages of each approach?
When would you choose one over the other?

In [124]:
raw_valid, rejected = extract_elt(raw_orders)
load_raw_to_lake(raw_valid, "e_commerce_lake.parquet")
clean_df_elt = transform_from_lake("e_commerce_lake.parquet")

print("Final rows after transformation:", len(clean_df_elt))

Extracted records: 200
Rejected records: 0
Raw data saved to: e_commerce_lake.parquet
Rows saved: 200
Final rows after transformation: 190


In ETL, after transformation we have 185 rows, and in ELT we have 190 rows. This is because in ELT we did not check price and quantity during extraction, and string values were converted to numbers instead of being dropped.
In ETL, data is checked and cleaned before it is loaded, so only valid records reach the destination. In ELT, most raw records are loaded first, and problems are fixed later during transformation.
ETL is useful when the destination system needs clean data. ELT is better when you want to store all raw data first and clean it later, which is common in data lakes and big analytics systems.

# Task 3: Simulate Modes of Dataflow
Model a simplified analytics system with three components: an order ingestion service, a feature computation service, and a prediction service.

### Step 1 — Data passing through a database: Simulate a shared database using a dictionary. The order service writes new orders to the database. The feature service reads orders and computes features (total orders, average amount, last order date per customer). The prediction service reads the features.

In [128]:
# Services communicate through a database
database = {"orders": [], "features": {}}
def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():
    orders = database["orders"]
    df = pd.DataFrame(orders)
    if df.empty:
        return
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["total_amount"] = df["quantity"] * df["unit_price"]
    features = df.groupby("customer_id").agg(
        total_orders=("order_id", "count"),
        avg_amount=("total_amount", "mean"),
        last_order_date=("order_date", "max")
    )
    database["features"] = features.to_dict(orient="index")

def prediction_service_read(customer_id):
    return database["features"].get(customer_id, "No features found")

order_service_write({
    "order_id": 1,
    "customer_id": 1001,
    "quantity": 2,
    "unit_price": 50,
    "order_date": "2025-03-01"
})
order_service_write({
    "order_id": 2,
    "customer_id": 1001,
    "quantity": 1,
    "unit_price": 30,
    "order_date": "2025-03-02"
})
feature_service_compute()
print(prediction_service_read(1001))

{'total_orders': 2, 'avg_amount': 65.0, 'last_order_date': Timestamp('2025-03-02 00:00:00')}


# Step 2 — Data passing through services: Refactor so the prediction service requests data from the feature service directly (function call simulating an API request). The feature service requests raw orders from the order service.

In [131]:
#Services talk directly
orders_db = []

def order_service_get_orders():
    return orders_db

def order_service_write(order):
    orders_db.append(order)

def feature_service_compute_direct():
    orders = order_service_get_orders()
    df = pd.DataFrame(orders)
    if df.empty:
        return {}
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["total_amount"] = df["quantity"] * df["unit_price"]
    features = df.groupby("customer_id").agg(
        total_orders=("order_id","count"),
        avg_amount=("total_amount","mean"),
        last_order_date=("order_date","max")
    )
    return features.to_dict(orient="index")

def prediction_service_read_direct(customer_id):
    features = feature_service_compute_direct()
    return features.get(customer_id, "No features found")

### Step 3 — Data passing through a message broker: Implement a simple pubsub broker (a class with publish, subscribe, and get_latest methods). The order service publishes new orders to an "orders" topic. The feature service subscribes and computes features whenever new orders arrive, publishing them to a "features" topic. The prediction service subscribes to "features."

In [134]:
class MessageBroker:
    def __init__(self):
        self.topics = {}
    def publish(self, topic, message):
        self.topics[topic] = message
    def get_latest(self, topic):
        return self.topics.get(topic, None)

broker = MessageBroker()

orders = []

def order_service_publish(order):
    orders.append(order)
    broker.publish("orders", orders)

def feature_service_compute_pub_sub():
    data = broker.get_latest("orders") or []
    df = pd.DataFrame(data)
    if df.empty:
        return
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["total_amount"] = df["quantity"] * df["unit_price"]
    features = df.groupby("customer_id").agg(
        total_orders=("order_id","count"),
        avg_amount=("total_amount","mean"),
        last_order_date=("order_date","max")
    )
    broker.publish("features", features.to_dict(orient="index"))

def prediction_service_read(customer_id):
    features = broker.get_latest("features") or {}
    return features.get(customer_id, "No features found")

### Step 4: Run 20 new orders through all three dataflow modes. Verify that the prediction service gets the same features in each case. Write a markdown cell comparing the three modes: coupling, latency characteristics, and what happens when one component goes down.

In [137]:
random.seed(42)
new_orders = []
for i in range(1, 21):
    new_orders.append({
        "order_id": i,
        "customer_id": random.randint(1000, 1005),
        "quantity": random.randint(1,5),
        "unit_price": round(random.uniform(10,200),2),
        "order_date": (datetime(2025,3,1) + timedelta(days=random.randint(0,10))).strftime("%Y-%m-%d")
    })
# We reset everything so that each dataflow mode processes the same fresh data. This makes the comparison fair and accurate.
database = {"orders": [], "features": {}} 
orders_db = [] 
broker = MessageBroker() 
broker.topics = {} 
orders_pubsub = []
# Mode 1: Database
database["orders"].clear()
for o in new_orders:
    database["orders"].append(o)
feature_service_compute()
db_features = database["features"]
print("Database Mode (customer 1001):", db_features.get(1001))

# Mode 2: Direct function calls
orders_db.clear()
for o in new_orders:
    orders_db.append(o)
direct_features = feature_service_compute_direct()
print("Direct Call Mode (customer 1001):", direct_features.get(1001))

# Mode 3: Pub/Sub
orders_pubsub.clear()
broker.topics.clear()
for o in new_orders:
    orders_pubsub.append(o)
    broker.publish("orders", orders_pubsub)
    # recompute features
    data = broker.get_latest("orders") or []
    df = pd.DataFrame(data)
    if not df.empty:
        df["order_date"] = pd.to_datetime(df["order_date"])
        df["total_amount"] = df["quantity"] * df["unit_price"]
        features = df.groupby("customer_id").agg(
            total_orders=("order_id","count"),
            avg_amount=("total_amount","mean"),
            last_order_date=("order_date","max")
        )
        broker.publish("features", features.to_dict(orient="index"))

pubsub_features = broker.get_latest("features") or {}
print("Pub/Sub Mode (customer 1001):", pubsub_features.get(1001))

Database Mode (customer 1001): {'total_orders': 5, 'avg_amount': 194.43, 'last_order_date': Timestamp('2025-03-11 00:00:00')}
Direct Call Mode (customer 1001): {'total_orders': 5, 'avg_amount': 194.43, 'last_order_date': Timestamp('2025-03-11 00:00:00')}
Pub/Sub Mode (customer 1001): {'total_orders': 5, 'avg_amount': 194.43, 'last_order_date': Timestamp('2025-03-11 00:00:00')}


The three primary modes of dataflow are through databases, through services, and through real-time transports. Using 20 new orders for customer 1001, we got the same features in all three modes:
Through databases: Services share a database; simple, but if the database fails, all stop.
Through services: Services call each other directly; fast but tightly connected, so failure of one affects others.
Through real-time transports (Pub/Sub): Services communicate via messages; flexible and loosely connected, can recover if one service is down.

# Task 4: Batch Processing vs Stream Processing
Step 1 — Batch processing: Using the cleaned data from Task 1, write a batch processing function that computes daily aggregate features:

Total orders per day
Total revenue per day
Average order size per day
Number of unique customers per day
Top product per day (by revenue)
Execute it and display the results.

In [141]:
def batch_daily_agg(df):
    df["order_date"] = pd.to_datetime(df["order_date"])
    daily = df.groupby(df["order_date"].dt.date).agg(
        total_orders=("order_id","count"),
        total_revenue=("total_amount","sum"),
        avg_order_size=("total_amount","mean"),
        unique_customers=("customer_id", pd.Series.nunique)
    ).reset_index().rename(columns={"order_date":"date"})

    top_products = []
    for date, group in df.groupby(df["order_date"].dt.date):
        top_product = group.groupby("product_name")["total_amount"].sum().idxmax()
        top_products.append(top_product)

    daily["top_product"] = top_products
    return daily

In [143]:
daily_features = batch_daily_agg(clean_df)
print(daily_features)

          date  total_orders  total_revenue  avg_order_size  unique_customers  \
0   2025-01-01             2         618.47      309.235000                 1   
1   2025-01-02             5        4282.19      856.438000                 5   
2   2025-01-03             3        3635.01     1211.670000                 3   
3   2025-01-04             2        2472.72     1236.360000                 2   
4   2025-01-05             6        3016.62      502.770000                 6   
5   2025-01-06             3         630.50      210.166667                 3   
6   2025-01-07             1        2496.30     2496.300000                 1   
7   2025-01-08             2        1778.88      889.440000                 2   
8   2025-01-09             2        1463.14      731.570000                 2   
9   2025-01-10             2        1247.63      623.815000                 2   
10  2025-01-11             2         420.94      210.470000                 2   
11  2025-01-12             3

In [145]:
daily_features.to_dict()

{'date': {0: datetime.date(2025, 1, 1),
  1: datetime.date(2025, 1, 2),
  2: datetime.date(2025, 1, 3),
  3: datetime.date(2025, 1, 4),
  4: datetime.date(2025, 1, 5),
  5: datetime.date(2025, 1, 6),
  6: datetime.date(2025, 1, 7),
  7: datetime.date(2025, 1, 8),
  8: datetime.date(2025, 1, 9),
  9: datetime.date(2025, 1, 10),
  10: datetime.date(2025, 1, 11),
  11: datetime.date(2025, 1, 12),
  12: datetime.date(2025, 1, 13),
  13: datetime.date(2025, 1, 15),
  14: datetime.date(2025, 1, 16),
  15: datetime.date(2025, 1, 17),
  16: datetime.date(2025, 1, 18),
  17: datetime.date(2025, 1, 19),
  18: datetime.date(2025, 1, 20),
  19: datetime.date(2025, 1, 21),
  20: datetime.date(2025, 1, 22),
  21: datetime.date(2025, 1, 23),
  22: datetime.date(2025, 1, 24),
  23: datetime.date(2025, 1, 25),
  24: datetime.date(2025, 1, 26),
  25: datetime.date(2025, 1, 27),
  26: datetime.date(2025, 1, 28),
  27: datetime.date(2025, 1, 29),
  28: datetime.date(2025, 1, 30),
  29: datetime.date(2025,

### Step 2 — Stream processing: Implement a StreamProcessor class that processes orders one at a time and maintains running statistics:

Running total of orders and revenue
Windowed average (last 50 orders) of order size
Running count of unique customers
Current most popular product (by count in the last 50 orders)
Process the same data from Task 1 through the stream processor, one record at a time. After every 50 records, print the current streaming statistics.

In [148]:
from collections import Counter

class StreamProcessor:
    def __init__(self, window_size=50):
        self.window_size = window_size
        self.total_orders = 0
        self.total_revenue = 0
        self.customers = set()
        self.last_orders = []

    def process_order(self, order):
        self.total_orders += 1
        self.total_revenue += order["total_amount"]
        self.customers.add(order["customer_id"])
        # keep last 50 orders
        self.last_orders.append(order)
        if len(self.last_orders) > self.window_size:
            self.last_orders.pop(0)

    def print_stats(self):
        if not self.last_orders:
            return
        avg_order = sum(o["total_amount"] for o in self.last_orders) / len(self.last_orders)
        products = [o["product_name"] for o in self.last_orders]
        most_popular = Counter(products).most_common(1)[0][0]
        print("Total orders:", self.total_orders)
        print("Total revenue:", round(self.total_revenue,2))
        print("Unique customers:", len(self.customers))
        print("Avg order size (last 50):", round(avg_order,2))
        print("Most popular product (last 50):", most_popular)
        print("-"*40)
        
stream = StreamProcessor()
for i, order in enumerate(clean_df.to_dict(orient="records"), 1):
    stream.process_order(order)
    if i % 50 == 0:
        print(f"--- After {i} orders ---")
        stream.print_stats()

--- After 50 orders ---
Total orders: 50
Total revenue: 33826.0
Unique customers: 39
Avg order size (last 50): 676.52
Most popular product (last 50): Textbooks
----------------------------------------
--- After 100 orders ---
Total orders: 100
Total revenue: 69294.14
Unique customers: 63
Avg order size (last 50): 709.36
Most popular product (last 50): Refrigerator
----------------------------------------
--- After 150 orders ---
Total orders: 150
Total revenue: 107674.99
Unique customers: 81
Avg order size (last 50): 767.62
Most popular product (last 50): Textbooks
----------------------------------------


### Step 3 — Compare final results: After processing all records through both approaches, compare the final totals. They should match for cumulative statistics (total orders, total revenue). Write a markdown cell explaining: Why might windowed streaming statistics differ from daily batch statistics even on the same data? What does each approach tell you that the other does not?

In [151]:
batch_total_orders = daily_features["total_orders"].sum()
batch_total_revenue = daily_features["total_revenue"].sum()

print("Batch total orders:", batch_total_orders)
print("Batch total revenue:", round(batch_total_revenue,2))

print("Stream total orders:", stream.total_orders)
print("Stream total revenue:", round(stream.total_revenue,2))

Batch total orders: 185
Batch total revenue: 135688.16
Stream total orders: 185
Stream total revenue: 135688.16


### Comparison of Batch vs Stream Processing
The final cumulative statistics from both approaches match: 185 total orders and 135,688.16 total revenue. This happens because both batch and streaming process the same dataset and calculate totals across all records.
However, windowed streaming statistics can differ from daily batch statistics. Streaming uses a moving window (for example, the last 50 orders), so metrics like average order size or most popular product reflect recent activity. Batch processing groups data by day and provides a full summary for each day.
Streaming is useful for real-time monitoring, while batch processing is better for historical analysis and daily reporting.

# Task 5: Combine Batch and Stream Features
Step 1: Define a set of batch features and streaming features for a customer in an e-commerce system:

Batch features (computed from all historical data):

total_lifetime_orders
avg_order_amount
days_since_first_order
most_purchased_category
Streaming features (computed from last 10 orders):

recent_order_count
recent_avg_amount
seconds_since_last_order
recent_top_category

In [155]:
# Example: select a single customer
customer_id = 1001
customer_orders = clean_df[clean_df["customer_id"] == customer_id].copy()
customer_orders["order_date"] = pd.to_datetime(customer_orders["order_date"])
customer_orders["total_amount"] = customer_orders["quantity"] * customer_orders["unit_price"]

# --- Batch features ---
total_lifetime_orders = len(customer_orders)
avg_order_amount = customer_orders["total_amount"].mean()
days_since_first_order = (datetime(2025,3,8) - customer_orders["order_date"].min()).days
most_purchased_category = customer_orders["product_name"].mode()[0]

batch_features = {
    "total_lifetime_orders": total_lifetime_orders,
    "avg_order_amount": round(avg_order_amount,2),
    "days_since_first_order": days_since_first_order,
    "most_purchased_category": most_purchased_category
}

# --- Streaming features (last 10 orders) ---
last_10_orders = customer_orders.tail(10)
recent_order_count = len(last_10_orders)
recent_avg_amount = last_10_orders["total_amount"].mean()
seconds_since_last_order = (datetime(2025,3,8) - last_10_orders["order_date"].max()).total_seconds()
recent_top_category = last_10_orders["product_name"].mode()[0]

streaming_features = {
    "recent_order_count": recent_order_count,
    "recent_avg_amount": round(recent_avg_amount,2),
    "seconds_since_last_order": int(seconds_since_last_order),
    "recent_top_category": recent_top_category
}

print("Batch features:", batch_features)
print("Streaming features:", streaming_features)

Batch features: {'total_lifetime_orders': 1, 'avg_order_amount': np.float64(2496.3), 'days_since_first_order': 60, 'most_purchased_category': 'Headphones'}
Streaming features: {'recent_order_count': 1, 'recent_avg_amount': np.float64(2496.3), 'seconds_since_last_order': 5184000, 'recent_top_category': 'Headphones'}


### Step 2: For 5 sample customers, compute both batch and streaming features from the Task 1 data.

In [158]:
# Pick 5 sample customers
sample_customers = clean_df["customer_id"].drop_duplicates().sample(5, random_state=42).tolist()

# -----------------------------
# Batch Features (all history)
# -----------------------------
batch_features = []

for cust in sample_customers:
    cust_orders = clean_df[clean_df["customer_id"] == cust].sort_values("order_date")
    
    total_orders = len(cust_orders)
    avg_amount = cust_orders["total_amount"].mean()
    days_since_first_order = (clean_df["order_date"].max() - cust_orders["order_date"].min()).days
    most_purchased_category = cust_orders["product_name"].mode()[0] if not cust_orders["product_name"].mode().empty else None
    
    batch_features.append({
        "Customer ID": cust,
        "Total Lifetime Orders": total_orders,
        "Avg Order Amount": round(avg_amount, 2),
        "Days Since First Order": days_since_first_order,
        "Most Purchased Category": most_purchased_category
    })

batch_df = pd.DataFrame(batch_features)
print("=== Batch Features ===")
print(batch_df)

# -----------------------------
# Streaming Features (last 10 orders)
# -----------------------------
stream_features = []

for cust in sample_customers:
    cust_orders = clean_df[clean_df["customer_id"] == cust].sort_values("order_date")
    last_orders = cust_orders.tail(10)
    
    recent_order_count = len(last_orders)
    recent_avg_amount = last_orders["total_amount"].mean()
    
    # Time since last order in seconds
    if len(last_orders) > 0:
        seconds_since_last_order = (clean_df["order_date"].max() - last_orders["order_date"].max()).total_seconds()
    else:
        seconds_since_last_order = None
    
    recent_top_category = last_orders["product_name"].mode()[0] if not last_orders["product_name"].mode().empty else None
    
    stream_features.append({
        "Customer ID": cust,
        "Recent Order Count": recent_order_count,
        "Recent Avg Amount": round(recent_avg_amount, 2),
        "Seconds Since Last Order": int(seconds_since_last_order) if seconds_since_last_order is not None else None,
        "Recent Top Category": recent_top_category
    })

stream_df = pd.DataFrame(stream_features)
print("\n=== Streaming Features ===")
print(stream_df)

=== Batch Features ===
   Customer ID  Total Lifetime Orders  Avg Order Amount  \
0         1068                      2            405.93   
1         1049                      1            284.54   
2         1069                      2           1103.67   
3         1046                      3            644.25   
4         1018                      1             80.16   

   Days Since First Order Most Purchased Category  
0                      44                  Tablet  
1                      15            Refrigerator  
2                      45                Sneakers  
3                      26               Textbooks  
4                      37              Headphones  

=== Streaming Features ===
   Customer ID  Recent Order Count  Recent Avg Amount  \
0         1068                   2             405.93   
1         1049                   1             284.54   
2         1069                   2            1103.67   
3         1046                   3             644.25 

### Step 3: Combine them into a single feature dictionary per customer. Print the combined feature vectors for all 5 customers.

In [161]:
# Combine batch and streaming features per customer
combined_features = []

for cust in sample_customers:
    # Get batch and stream feature dicts for this customer
    batch_row = batch_df[batch_df["Customer ID"] == cust].to_dict(orient="records")[0]
    stream_row = stream_df[stream_df["Customer ID"] == cust].to_dict(orient="records")[0]
    
    # Merge dictionaries
    combined = {**batch_row, **stream_row}
    combined_features.append(combined)

# Print combined feature vectors
print("=== Combined Features ===")
for feat in combined_features:
    print(feat)

=== Combined Features ===
{'Customer ID': 1068, 'Total Lifetime Orders': 2, 'Avg Order Amount': 405.93, 'Days Since First Order': 44, 'Most Purchased Category': 'Tablet', 'Recent Order Count': 2, 'Recent Avg Amount': 405.93, 'Seconds Since Last Order': 86400, 'Recent Top Category': 'Tablet'}
{'Customer ID': 1049, 'Total Lifetime Orders': 1, 'Avg Order Amount': 284.54, 'Days Since First Order': 15, 'Most Purchased Category': 'Refrigerator', 'Recent Order Count': 1, 'Recent Avg Amount': 284.54, 'Seconds Since Last Order': 1296000, 'Recent Top Category': 'Refrigerator'}
{'Customer ID': 1069, 'Total Lifetime Orders': 2, 'Avg Order Amount': 1103.67, 'Days Since First Order': 45, 'Most Purchased Category': 'Sneakers', 'Recent Order Count': 2, 'Recent Avg Amount': 1103.67, 'Seconds Since Last Order': 3024000, 'Recent Top Category': 'Sneakers'}
{'Customer ID': 1046, 'Total Lifetime Orders': 3, 'Avg Order Amount': 644.25, 'Days Since First Order': 26, 'Most Purchased Category': 'Textbooks', 'Re

### Step 4: Write a markdown cell explaining why an ML model would benefit from having both batch and stream features. Give an example of a prediction task where the batch features alone would be insufficient, and another where the stream features alone would be insufficient.

Machine learning models benefit from both because batch features capture long-term patterns (like total orders or favorite category), while stream features capture recent activity (like the last 10 orders). For example, predicting if a customer will buy today needs recent behavior, so batch alone isn’t enough, while recommending a favorite product needs long-term trends, so stream alone isn’t enough.